# Dominanța naturii

**Florile și copacii dau mai multe nume decât toți voievozii și poeții la un loc.**

Natura (flori, copaci, apă, geografie) reprezintă 54% din totalul străzilor clasificate — mai mult decât persoanele (23%). Acest notebook explorează ce tipuri de natură domină și cum variază regional.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.family':'serif','figure.dpi':130,
                     'axes.spines.top':False,'axes.spines.right':False})
ACCENT, INK, MUTED = '#C04F35', '#15171A', '#6E6E70'

NATURE_RO = {
    'flower':'flori','tree':'copaci','water':'apă','plant':'plante',
    'bird':'păsări','sky':'cer','geography':'geografie','forest':'pădure',
    'meadow':'câmpie','season':'anotimpuri','animal':'animale',
    'weather':'vreme','fruit':'fructe','field':'câmp','hill':'deal',
    'orchard':'livadă','valley':'vale','mountain':'munte',
    'mountain_peak':'vârf de munte','peak':'vârf'
}

conn = sqlite3.connect('../data/streets.db')
conn.row_factory = sqlite3.Row
print('Connected.')

## 1. Subtipuri de natură

In [ ]:
df_sub = pd.read_sql("""
    SELECT nt.nature_type, COUNT(*) AS streets
    FROM electoral_dedup sd
    JOIN nature_terms nt ON nt.core_name_norm = sd.core_name_norm
    GROUP BY nt.nature_type
    ORDER BY streets DESC
""", conn)

df_sub['label'] = df_sub['nature_type'].map(NATURE_RO).fillna(df_sub['nature_type'])

colors = ['#A0826A','#8BA888','#6E8FAA','#B8A090','#8A9A7E','#7A8A9A',
          '#C09870','#6A8068','#9A7860','#A88888','#7888A0','#888898',
          '#B09078','#90A088','#887870']

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_sub['label'], df_sub['streets'],
       color=colors[:len(df_sub)], edgecolor='white')
ax.set_title('Distribuția subtipurilor de natură', fontsize=13)
ax.set_ylabel('Număr de străzi')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'.replace(',','.')))
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 2. Top termeni per subtip

In [ ]:
for subtype in ['flower','tree','bird','water']:
    df_t = pd.read_sql("""
        SELECT nt.term, COUNT(*) AS streets
        FROM electoral_dedup sd
        JOIN nature_terms nt ON nt.core_name_norm = sd.core_name_norm
        WHERE nt.nature_type = ?
        GROUP BY nt.core_name_norm
        ORDER BY streets DESC
        LIMIT 10
    """, conn, params=(subtype,))
    label = NATURE_RO.get(subtype, subtype)
    print(f'\n--- {label.upper()} ---')
    print(df_t.to_string(index=False))

## 3. Natură vs. persoane pe județ

In [ ]:
df_jud = pd.read_sql("""
    SELECT sd.judet,
        SUM(CASE WHEN nt.core_name_norm IS NOT NULL THEN 1 ELSE 0 END) AS nature_streets,
        SUM(CASE WHEN p.core_name_norm IS NOT NULL THEN 1 ELSE 0 END) AS person_streets,
        COUNT(*) AS total
    FROM electoral_dedup sd
    LEFT JOIN nature_terms nt ON nt.core_name_norm = sd.core_name_norm
    LEFT JOIN persons p ON p.core_name_norm = sd.core_name_norm
    GROUP BY sd.judet
    ORDER BY nature_streets DESC
""", conn)

df_jud['nature_pct'] = df_jud['nature_streets'] / df_jud['total'] * 100
df_jud['person_pct'] = df_jud['person_streets'] / df_jud['total'] * 100

fig, ax = plt.subplots(figsize=(13, 5))
x = range(len(df_jud))
ax.bar(x, df_jud['nature_pct'], label='Natură', color='#8BA888', alpha=0.85)
ax.bar(x, df_jud['person_pct'], bottom=df_jud['nature_pct'], label='Persoane', color=INK, alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(df_jud['judet'], rotation=45, ha='right')
ax.set_title('Natură vs. persoane din totalul județean clasificat', fontsize=13)
ax.set_ylabel('%')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
conn.close()